# Relative Position Attention (相对位置注意力) 实现

相对位置注意力编码token之间的相对位置关系，而非绝对位置。

**核心思想**：
- 使用相对位置而非绝对位置编码
- 捕获token之间的相对距离关系
- 更好的长度泛化能力

**公式**：
$$
\text{score}(i,j) = \frac{Q_i \cdot K_j + Q_i \cdot R_{i-j}}{\sqrt{d_k}}
$$
其中$R_{i-j}$是相对位置编码，只依赖于相对距离$i-j$。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_style('whitegrid')

## 1. 辅助函数

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def get_relative_positions(seq_len, max_relative_position=None):
    """
    生成相对位置矩阵
    
    Args:
        seq_len: 序列长度
        max_relative_position: 最大相对位置距离
    
    Returns:
        相对位置矩阵 (seq_len, seq_len)
    """
    if max_relative_position is None:
        max_relative_position = seq_len - 1
    
    range_vec = np.arange(seq_len)
    relative_positions = range_vec[None, :] - range_vec[:, None]
    
    # 裁剪到[-max_relative_position, max_relative_position]
    relative_positions = np.clip(
        relative_positions,
        -max_relative_position,
        max_relative_position
    )
    
    return relative_positions


def relative_position_to_index(relative_positions, max_relative_position):
    """将相对位置转换为嵌入索引"""
    indices = relative_positions + max_relative_position
    return indices

## 2. 可视化相对位置矩阵

In [ ]:
# 生成相对位置矩阵
seq_len = 8
max_relative_position = 4

relative_pos = get_relative_positions(seq_len, max_relative_position)

print(f"相对位置矩阵 (seq_len={seq_len}, max={max_relative_position}):\n")
print(relative_pos)

# 可视化
plt.figure(figsize=(10, 8))
sns.heatmap(relative_pos, annot=True, fmt='d', cmap='RdBu_r', 
            center=0, cbar_kws={'label': '相对距离'})
plt.title('相对位置矩阵', fontsize=14, fontweight='bold')
plt.xlabel('Key位置', fontsize=12)
plt.ylabel('Query位置', fontsize=12)
plt.tight_layout()
plt.show()

print("\n说明:")
print("  • 矩阵[i,j]表示从位置i到位置j的相对距离")
print("  • 正值: j在i之后; 负值: j在i之前")
print("  • 对角线为0（自身到自身）")
print(f"  • 超过±{max_relative_position}的距离被裁剪")

## 3. 实现Relative Position Attention类

In [ ]:
class RelativePositionAttention:
    """
    Relative Position Attention (相对位置注意力)
    
    使用相对位置编码而非绝对位置编码
    """
    
    def __init__(self, embed_dim, num_heads, max_relative_position=None, use_bias=True):
        assert embed_dim % num_heads == 0
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.max_relative_position = max_relative_position
        self.use_bias = use_bias
        
        # Q, K, V投影
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_o = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        if use_bias:
            self.b_q = np.zeros(embed_dim)
            self.b_k = np.zeros(embed_dim)
            self.b_v = np.zeros(embed_dim)
            self.b_o = np.zeros(embed_dim)
        
        # 相对位置嵌入
        if max_relative_position is not None:
            num_relative_positions = 2 * max_relative_position + 1
            self.relative_position_embeddings = np.random.randn(
                num_heads, num_relative_positions, self.head_dim
            ) / np.sqrt(self.head_dim)
        else:
            self.relative_position_embeddings = None
    
    def _get_relative_embeddings(self, seq_len):
        """获取或创建相对位置嵌入"""
        max_relative_position = self.max_relative_position
        if max_relative_position is None:
            max_relative_position = seq_len - 1
        
        if self.relative_position_embeddings is None:
            num_relative_positions = 2 * max_relative_position + 1
            relative_embeddings = np.random.randn(
                self.num_heads, num_relative_positions, self.head_dim
            ) / np.sqrt(self.head_dim)
        else:
            relative_embeddings = self.relative_position_embeddings
        
        return relative_embeddings, max_relative_position
    
    def split_heads(self, x):
        """分割成多个头"""
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 0, 2)
        return x
    
    def combine_heads(self, x):
        """合并多个头"""
        x = x.transpose(1, 0, 2)
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.embed_dim)
        return x
    
    def compute_attention_with_relative_positions(self, Q, K, V, relative_embeddings,
                                                   relative_positions, mask=None):
        """计算带相对位置编码的注意力"""
        num_heads, seq_len_q, head_dim = Q.shape
        seq_len_k = K.shape[1]
        
        # 1. 内容得分: Q @ K^T
        content_scores = np.matmul(Q, K.transpose(0, 2, 1))
        
        # 2. 相对位置得分
        max_relative_position = (relative_embeddings.shape[1] - 1) // 2
        relative_indices = relative_position_to_index(relative_positions, max_relative_position)
        
        position_scores = np.zeros((num_heads, seq_len_q, seq_len_k))
        for h in range(num_heads):
            for i in range(seq_len_q):
                for j in range(seq_len_k):
                    rel_idx = relative_indices[i, j]
                    position_scores[h, i, j] = np.dot(Q[h, i], relative_embeddings[h, rel_idx])
        
        # 3. 合并得分
        scores = (content_scores + position_scores) / np.sqrt(head_dim)
        
        # 4. 应用mask
        if mask is not None:
            if mask.ndim == 2:
                mask = mask[None, :, :]
            scores = np.where(mask == 0, -1e9, scores)
        
        # 5. Softmax
        attention_weights = softmax(scores, axis=-1)
        
        # 6. 加权求和
        output = np.matmul(attention_weights, V)
        
        return output, attention_weights
    
    def forward(self, query, key=None, value=None, mask=None, return_attention=False):
        """前向传播"""
        if key is None:
            key = query
        if value is None:
            value = key
        
        seq_len_q = query.shape[0]
        seq_len_k = key.shape[0]
        
        # 1. 线性投影
        Q = np.dot(query, self.W_q)
        K = np.dot(key, self.W_k)
        V = np.dot(value, self.W_v)
        
        if self.use_bias:
            Q += self.b_q
            K += self.b_k
            V += self.b_v
        
        # 2. 分割头
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # 3. 获取相对位置信息
        relative_embeddings, max_relative_position = self._get_relative_embeddings(
            max(seq_len_q, seq_len_k)
        )
        relative_positions = get_relative_positions(seq_len_q, max_relative_position)
        if seq_len_q != seq_len_k:
            relative_positions = relative_positions[:, :seq_len_k]
        
        # 4. 计算注意力
        multi_head_output, attention_weights = self.compute_attention_with_relative_positions(
            Q, K, V, relative_embeddings, relative_positions, mask=mask
        )
        
        # 5. 合并头
        concatenated = self.combine_heads(multi_head_output)
        
        # 6. 输出投影
        output = np.dot(concatenated, self.W_o)
        if self.use_bias:
            output += self.b_o
        
        if return_attention:
            return output, attention_weights
        
        return output


class RelativePositionSelfAttention(RelativePositionAttention):
    """相对位置自注意力（简化接口）"""
    
    def forward(self, x, mask=None, return_attention=False):
        return super().forward(x, x, x, mask=mask, return_attention=return_attention)

## 4. 测试相对位置注意力

In [ ]:
# 参数设置
seq_len = 10
embed_dim = 64
num_heads = 4
max_relative_position = 4

print("配置:")
print(f"  序列长度: {seq_len}")
print(f"  嵌入维度: {embed_dim}")
print(f"  注意力头数: {num_heads}")
print(f"  最大相对位置: {max_relative_position}")

# 生成输入
x = np.random.randn(seq_len, embed_dim)

# 创建相对位置注意力层
rpa = RelativePositionSelfAttention(embed_dim, num_heads, max_relative_position)

# 前向传播
output, attn_weights = rpa.forward(x, return_attention=True)

print(f"\n形状:")
print(f"  输入: {x.shape}")
print(f"  输出: {output.shape}")
print(f"  注意力权重: {attn_weights.shape}")

## 5. 可视化注意力模式

In [ ]:
# 可视化所有头的注意力
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i in range(4):
    sns.heatmap(attn_weights[i], annot=True, fmt='.2f', cmap='YlOrRd',
                ax=axes[i], cbar=True, square=True, vmin=0, vmax=0.3)
    axes[i].set_title(f'注意力头 {i+1}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Key位置')
    axes[i].set_ylabel('Query位置')

plt.tight_layout()
plt.show()

print("观察:")
print("  • 每个头学习到不同的注意力模式")
print("  • 相对位置影响注意力分布")
print("  • 每行和为1.0（softmax归一化）")

## 6. 分析相对位置的影响

In [ ]:
# 选择一个query位置，分析不同相对距离的影响
query_pos = 5
head_idx = 0

# 获取该query位置的注意力分布
attention_dist = attn_weights[head_idx, query_pos, :]

# 计算相对距离
key_positions = np.arange(seq_len)
relative_distances = key_positions - query_pos

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：注意力权重 vs Key位置
ax1.bar(key_positions, attention_dist, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axvline(query_pos, color='red', linestyle='--', linewidth=2, label=f'Query位置={query_pos}')
ax1.set_xlabel('Key位置', fontsize=12)
ax1.set_ylabel('注意力权重', fontsize=12)
ax1.set_title(f'Query位置{query_pos}的注意力分布', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 右图：注意力权重 vs 相对距离
sorted_idx = np.argsort(relative_distances)
sorted_distances = relative_distances[sorted_idx]
sorted_attention = attention_dist[sorted_idx]

colors = ['green' if d < 0 else 'red' if d > 0 else 'blue' for d in sorted_distances]
ax2.bar(sorted_distances, sorted_attention, color=colors, alpha=0.7, edgecolor='black')
ax2.set_xlabel('相对距离（负=之前，正=之后）', fontsize=12)
ax2.set_ylabel('注意力权重', fontsize=12)
ax2.set_title('注意力 vs 相对距离', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Query位置{query_pos}的注意力分析:")
print(f"\n{'Key位置':>10} {'相对距离':>10} {'注意力权重':>12}")
print("-" * 35)
for key_pos in range(seq_len):
    rel_dist = key_pos - query_pos
    attn = attention_dist[key_pos]
    print(f"{key_pos:>10} {rel_dist:>10} {attn:>12.4f}")

## 7. 因果Mask下的相对位置注意力

In [ ]:
def create_causal_mask(seq_len):
    """创建因果mask"""
    return np.tril(np.ones((seq_len, seq_len)))

# 应用因果mask
causal_mask = create_causal_mask(seq_len)
output_masked, attn_masked = rpa.forward(x, mask=causal_mask, return_attention=True)

# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 因果mask
sns.heatmap(causal_mask, annot=False, cmap='Greys',
            ax=axes[0], cbar=False, square=True)
axes[0].set_title('因果Mask', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Key位置')
axes[0].set_ylabel('Query位置')

# 无mask
sns.heatmap(attn_weights[0], annot=True, fmt='.2f', cmap='YlOrRd',
            ax=axes[1], square=True)
axes[1].set_title('头1 (无Mask)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Key位置')
axes[1].set_ylabel('Query位置')

# 有mask
sns.heatmap(attn_masked[0], annot=True, fmt='.2f', cmap='YlOrRd',
            ax=axes[2], square=True)
axes[2].set_title('头1 (带因果Mask)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Key位置')
axes[2].set_ylabel('Query位置')

plt.tight_layout()
plt.show()

print("因果Mask效果:")
print("  ✓ 每个位置只能看到自己和之前的位置")
print("  ✓ 用于GPT等自回归模型")
print("  ✓ 相对位置注意力 + 因果mask = 强大的自回归建模")

## 8. 长度泛化能力测试

In [ ]:
# 测试不同长度的序列
test_lengths = [8, 12, 16, 20]

print("长度泛化能力测试:")
print(f"\n训练配置: max_relative_position={max_relative_position}\n")
print(f"{'序列长度':>10} {'输入形状':>15} {'输出形状':>15} {'状态':>10}")
print("-" * 55)

for length in test_lengths:
    x_test = np.random.randn(length, embed_dim)
    output_test = rpa.forward(x_test)
    status = "✓" if output_test.shape == x_test.shape else "✗"
    print(f"{length:>10} {str(x_test.shape):>15} {str(output_test.shape):>15} {status:>10}")

print("\n优势:")
print("  ✓ 可以处理任意长度的序列")
print("  ✓ max_relative_position限制参数量")
print("  ✓ 超过max_relative_position的距离共享相同嵌入")
print("  ✓ 比绝对位置编码有更好的长度泛化能力")

## 9. 参数量分析

In [ ]:
# 计算参数量
def count_parameters(model):
    """计算模型参数量"""
    params = {
        'W_q': model.embed_dim * model.embed_dim,
        'W_k': model.embed_dim * model.embed_dim,
        'W_v': model.embed_dim * model.embed_dim,
        'W_o': model.embed_dim * model.embed_dim,
    }
    
    if model.use_bias:
        params['biases'] = 4 * model.embed_dim
    
    if model.relative_position_embeddings is not None:
        num_rel_pos = model.relative_position_embeddings.shape[1]
        params['relative_pos_emb'] = model.num_heads * num_rel_pos * model.head_dim
    
    return params

params = count_parameters(rpa)

print("相对位置注意力参数量分析:")
print(f"\n配置: embed_dim={embed_dim}, num_heads={num_heads}, max_relative_position={max_relative_position}\n")
print(f"{'参数':>20} {'数量':>15}")
print("-" * 38)
total = 0
for name, count in params.items():
    print(f"{name:>20} {count:>15,}")
    total += count
print("-" * 38)
print(f"{'总计':>20} {total:>15,}")

# 相对位置嵌入占比
if 'relative_pos_emb' in params:
    rel_pos_ratio = params['relative_pos_emb'] / total * 100
    print(f"\n相对位置嵌入占比: {rel_pos_ratio:.2f}%")

# 对比不同max_relative_position
print("\n不同max_relative_position的参数量对比:")
print(f"\n{'max_relative_position':>25} {'相对位置嵌入参数':>20} {'总参数':>15}")
print("-" * 65)

for max_rel in [4, 8, 16, 32, 64]:
    num_rel_pos = 2 * max_rel + 1
    rel_params = num_heads * num_rel_pos * (embed_dim // num_heads)
    total_params = total - params.get('relative_pos_emb', 0) + rel_params
    print(f"{max_rel:>25} {rel_params:>20,} {total_params:>15,}")

print("\n观察:")
print("  • max_relative_position越大，相对位置嵌入参数越多")
print("  • 但参数量与序列长度无关（vs 绝对位置编码）")
print("  • T5通常使用max_relative_position=32或128")

## 10. 实际应用：T5配置

In [ ]:
# T5-Base配置
t5_embed_dim = 768
t5_num_heads = 12
t5_max_relative_position = 32  # T5使用的配置

t5_rpa = RelativePositionSelfAttention(
    embed_dim=t5_embed_dim,
    num_heads=t5_num_heads,
    max_relative_position=t5_max_relative_position
)

print("T5-Base 相对位置注意力配置:")
print(f"\n  嵌入维度: {t5_embed_dim}")
print(f"  注意力头数: {t5_num_heads}")
print(f"  每个头维度: {t5_embed_dim // t5_num_heads}")
print(f"  最大相对位置: {t5_max_relative_position}")

# 相对位置嵌入
num_relative_embeddings = 2 * t5_max_relative_position + 1
print(f"\n  相对位置嵌入数量: {num_relative_embeddings}")
print(f"  相对位置嵌入参数: {t5_num_heads * num_relative_embeddings * (t5_embed_dim // t5_num_heads):,}")

# 测试不同序列长度
print("\n测试T5在不同序列长度下的性能:")
test_seq_lens = [64, 128, 256, 512]

for seq_len_test in test_seq_lens:
    x_test = np.random.randn(seq_len_test, t5_embed_dim)
    output_test = t5_rpa.forward(x_test)
    print(f"  序列长度 {seq_len_test:>3}: 输入{x_test.shape} -> 输出{output_test.shape} ✓")

print("\nT5的设计选择:")
print("  ✓ max_relative_position=32足以捕获大多数有用的相对位置信息")
print("  ✓ 超过32的距离共享相同的嵌入")
print("  ✓ 参数效率高，同时保持长距离建模能力")
print("  ✓ 可以处理任意长度的序列（不受训练长度限制）")

## 11. 相对位置 vs 绝对位置对比

In [ ]:
# 可视化对比
comparison_data = {
    '特性': ['位置编码方式', '长度泛化', '参数量', '训练复杂度', '推理灵活性'],
    '相对位置': ['相对距离', '优秀 ✓', '固定（与序列长度无关）', '稍高', '支持任意长度'],
    '绝对位置': ['绝对索引', '较差 ✗', '随序列长度增长', '简单', '受训练长度限制']
}

import pandas as pd
df = pd.DataFrame(comparison_data)

print("相对位置 vs 绝对位置编码对比:\n")
print(df.to_string(index=False))

print("\n\n相对位置注意力的优势:")
print("  ✓ 位置不变性：模式只依赖相对距离")
print("  ✓ 长度泛化：可处理比训练时更长的序列")
print("  ✓ 参数效率：使用max_relative_position限制参数量")
print("  ✓ 自然建模：语言理解更依赖相对位置")

print("\n应用实例:")
print("  • T5: 使用相对位置注意力作为核心组件")
print("  • Transformer-XL: 相对位置编码处理长序列")
print("  • DeBERTa: 解耦的相对位置注意力")
print("  • XLNet: 相对位置编码 + 排列语言模型")

## 总结

### 相对位置注意力的核心要点：

1. **核心机制**：
   - 编码token之间的相对距离，而非绝对位置
   - 注意力得分 = 内容得分 + 相对位置得分
   - 使用max_relative_position限制参数量

2. **关键优势**：
   - 更好的长度泛化能力
   - 参数量与序列长度无关
   - 更自然的语言建模方式
   - 位置不变性

3. **实际应用**：
   - T5使用相对位置注意力作为核心
   - Transformer-XL处理长序列
   - DeBERTa等改进模型

4. **设计选择**：
   - max_relative_position通常为32-128
   - 平衡参数量和建模能力
   - 超过max的距离共享嵌入

### 公式总结：
$$
\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T + \text{PositionScores}}{\sqrt{d_k}}\right)V
$$

其中PositionScores基于相对距离$i-j$计算。